In [ ]:
%cd ..

In [ ]:
from app.imc2025.prediction import load_from_train


samples = load_from_train("./data")

In [ ]:
from mts.helpers.project.project import Project


last_project_iteration = Project.from_next_iteration("iterations")

In [ ]:
dataset_name = "imc2023_haiper"

In [ ]:
repositories_dirpath = last_project_iteration.iteration_dirpath / "h5_repositories"

In [ ]:
repositories_dirpath.mkdir(exist_ok=True, parents=True)

In [ ]:
from mts.pipeline.repository import h5 as h5_repo

In [ ]:
image_repository = h5_repo.H5ImageRepository.from_filename(
    repositories_dirpath, dataset_name
)
image_repository.add_repository_metadata(dataset_name=dataset_name)

In [ ]:
for prediction in samples["imc2023_haiper"]:
    image_repository.add_image(prediction.image_filepath)

In [ ]:
import torch
from mts.core.model.mast3r.io import load_model

local_model_directory = (
    "checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
)

mast3r_model = load_model(local_model_directory, torch.device("cpu"))

cuda0 = torch.device("cuda:0")

mast3r_model = mast3r_model.to(cuda0)

In [ ]:
ids = [6, 7, 8, 14]

In [ ]:
from hloc.utils import viz as img_viz

img_viz.plot_images([image_repository.load_image(img_id) for img_id in ids])

In [ ]:
ids = list(range(image_repository.images_num()))

In [ ]:
filepaths = [image_repository.get_filepath(img_id) for img_id in ids]

In [ ]:
image_repository._image_id_to_filepath

In [ ]:
import numpy as np

In [ ]:
filepath_map_to_idx = {filepath: num for num, filepath in enumerate(filepaths)}
idx_to_filepath_map = {num: filepath for num, filepath in enumerate(filepaths)}
filepaths_as_str = list(map(str, filepaths))
idx_to_filepath_as_str_map = dict(zip(ids, filepaths_as_str))
filepath_as_str_to_ids_map = dict(zip(filepaths_as_str, ids))

filepath_map_to_id = dict(zip(filepaths, ids))

id_to_filepath_map = dict(zip(ids, filepaths))

In [ ]:
import more_itertools as mit

In [ ]:
from mts.core.matcher.dense.mast3r import match_pairs


keypoints_map, matches_map = match_pairs(
    mast3r_model,
    list(mit.pairwise(range(len(filepaths_as_str)))),
    filepaths_as_str,
    device=cuda0,
)

In [ ]:
local_model_directory = (
    "checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
)
retrival_model_dir = "checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_trainingfree.pth"

In [ ]:
scene_graph: str = "retrieval-20-40"

In [ ]:
from mts.pipeline.step.pair.mast3r import Mast3rParer


mast3r_parer = Mast3rParer.from_checkpoints(
    local_model_directory,
    retrival_model_dir,
    scene_graph,
)

In [ ]:
with torch.no_grad():
    similarity_matrix = mast3r_parer.retriever(filepaths_as_str)

In [ ]:
from plotly import express as px

In [ ]:
px.imshow(distance_matrix)

In [ ]:
from scipy.sparse.csgraph import minimum_spanning_tree

mst = minimum_spanning_tree(distance_matrix).toarray()

In [ ]:
import networkx as nx

In [ ]:
def graph_from_distance_matrix(
    matrix,
    labels: list[str],
    threshold: float = 1,
):
    n = matrix.shape[0]
    G = nx.Graph()

    # add nodes
    if labels is None:
        labels = list(range(n))
    G.add_nodes_from(labels)

    # add weighted edges
    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i, j] <= threshold:
                G.add_edge(labels[i], labels[j], weight=matrix[i, j])
    return G

In [ ]:
from mts.core.types import PairType


def pairs_from_distance_matrix(
    distance_matrix: np.ndarray,
    threshold: float = 0.99,
) -> list[PairType[int]]:
    distance_mask = distance_matrix <= threshold

    possible_pairs = []
    for num, (sim_row, sim_mask_row) in enumerate(zip(distance_matrix, distance_mask)):
        sorted_indices = np.argsort(sim_row)
        sorted_mask = sim_mask_row[sorted_indices]
        for idx in sorted_indices[sorted_mask]:
            if num == idx:
                continue
            possible_pairs.append(tuple(sorted([num, int(idx)])))
    return possible_pairs

In [ ]:
distance_mask = distance_matrix <= 0.99

In [ ]:
possible_pairs = []
for num, (sim_row, sim_mask_row) in enumerate(zip(distance_matrix, distance_mask)):
    sorted_indices = np.argsort(sim_row)
    sorted_mask = sim_mask_row[sorted_indices]
    for idx in sorted_indices[sorted_mask]:
        if num == idx:
            continue
        possible_pairs.append(tuple(sorted([num, int(idx)])))

In [ ]:
len(possible_pairs)

In [ ]:
from numpy import ma

In [ ]:
similarity_graph = graph_from_distance_matrix(
    distance_matrix,
    filepaths_as_str,
    1.01,
)

In [ ]:
mst = nx.minimum_spanning_tree(similarity_graph, weight="weight")

In [ ]:
matches_graph = nx.Graph()

In [ ]:
mst_pairs = []
for st_node, nd_node in mst.edges:
    st_node, nd_node = sorted((st_node, nd_node))
    mst_pairs.append(
        (
            filepath_as_str_to_ids_map[st_node],
            filepath_as_str_to_ids_map[nd_node],
        )
    )

In [ ]:
keypoints_map, matches_map = match_pairs(
    mast3r_model,
    mst_pairs,
    filepaths_as_str,
    device=cuda0,
)

In [ ]:
from mts.viz.plotly.nx.graph import plot_graph


plot_graph(mst)

In [ ]:
def mst_from_matrix_nx(
    matrix,
    labels: list[str],
    threshold: float = 0.95,
):
    n = matrix.shape[0]
    G = nx.Graph()

    # add nodes
    if labels is None:
        labels = list(range(n))
    G.add_nodes_from(labels)

    # add weighted edges
    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i, j] < threshold:
                G.add_edge(labels[i], labels[j], weight=matrix[i, j])

    # compute MST
    mst = nx.minimum_spanning_tree(G, weight="weight")

    return mst

In [ ]:
from app.imc2025.run.from_config import create_imc2025_from_cfg
from omegaconf import OmegaConf
from config.logging import setup, DEBUG

setup(DEBUG)
cfg = OmegaConf.load("config/pipeline/imc2025/0002-pairs.yaml")

In [ ]:
pipeline = create_imc2025_from_cfg(cfg)

In [ ]:
pipeline.run_for("imc2023_haiper")

In [ ]:
dataset_name = "imc2023_haiper"

In [ ]:
# TODO: include per image matches, not as concatenated

In [ ]:
pairs_idxs = []
for st_img_id, nd_img_id in pipeline._repositories_map[dataset_name].get_pairs():
    st_filepath = pipeline._repositories_map[dataset_name].get_filepath(st_img_id)
    nd_filepath = pipeline._repositories_map[dataset_name].get_filepath(nd_img_id)
    pairs_idxs.append(
        (filepath_map_to_idx[st_filepath], filepath_map_to_idx[nd_filepath])
    )

In [ ]:
from pathlib import Path

In [ ]:
import logging


LOGGER = logging.getLogger(__name__)

In [ ]:
from hloc.utils import viz as img_viz

kpts = map_to_kpts(
    keypoints_map,
    ids,
    id_to_filepath_map,
)
img_viz.plot_images(
    [
        image_repository.load_image(
            img_id,
        )
        for img_id in ids
    ],
)
plot_keypoints(kpts)

In [ ]:
from mts.core.matcher.dense.mast3r import extract_dense_keypoints

In [ ]:
len(mst_pairs)

In [ ]:
# TODO: check which edge is missing after comping the matches

In [ ]:
import more_itertools as mit

pairwise_idxs = list(mit.pairwise(range(len(filepaths))))
matches_map = extract_dense_keypoints(
    mast3r_model,
    mst_pairs,
    filepaths_as_str,
    device=cuda0,
)

In [ ]:
sum(map(len, matches_map.values()))

In [ ]:
from scipy.spatial import KDTree
import numpy as np

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from numpy import ma as np_ma

In [ ]:
from enum import Enum
from dataclasses import dataclass


@dataclass
class Image:
    height: int
    width: int

    @property
    def hw(self) -> tuple[int, int]:
        return self.height, self.width


class MatchKind(Enum):
    MATCHED = "Matched"
    MERGED = "Merged"

@dataclass
class TwoViewEdge:
    st_filepath: str
    nd_filepath: str
    kpts_for: dict[str, np.ndarray]
    match_kind: MatchKind
    num_matches: int

In [ ]:
import pycolmap


def validate_matches(
    st_kpts: np.ndarray,
    nd_kpts: np.ndarray,
    matches: np.ndarray,
    st_hw: tuple[int, int],
    nd_hw: tuple[int, int],
) -> tuple[np.ndarray, np.ndarray]:
    st_h, st_w = st_hw
    nd_h, nd_w = nd_hw

    camera1 = pycolmap.Camera(
        model="SIMPLE_PINHOLE",
        width=st_w,
        height=st_h,
        params=[0.9 * max(st_w, st_h), st_w / 2, st_h / 2],
    )

    camera2 = pycolmap.Camera(
        model="SIMPLE_PINHOLE",
        width=nd_w,
        height=nd_h,
        params=[0.9 * max(nd_w, nd_h), nd_w / 2, nd_h / 2],
    )

    options = pycolmap.TwoViewGeometryOptions()
    options.compute_relative_pose = True

    result = pycolmap.estimate_two_view_geometry(
        camera1,
        st_kpts,
        camera2,
        nd_kpts,
        matches=matches,
        options=options,
    )

    return result.inlier_matches


def validate_kps_matches(
    st_kpts: np.ndarray,
    nd_kpts: np.ndarray,
    st_hw: tuple[int, int],
    nd_hw: tuple[int, int],
) -> tuple[np.ndarray, np.ndarray]:
    arranged_matches = np.tile(np.arange(0, len(st_kpts))[:, np.newaxis], (1, 2))
    inlier_matches = validate_matches(
        st_kpts,
        nd_kpts,
        arranged_matches,
        st_hw,
        nd_hw,
    )
    return inlier_matches


def match_kpts(
    kpts_intermediary_from: np.ndarray,
    kpts_intermediary_to: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    nd_prev_kpts_tree = KDTree(kpts_intermediary_from)
    distance, indices = nd_prev_kpts_tree.query(kpts_intermediary_to, p=2)

    ma_indices = np_ma.masked_where(distance > 1, indices)

    from_idx, to_idx = np.unique(
        ma_indices,
        return_index=True,
    )
    indices_from = from_idx.compressed()
    indices_to = to_idx[~from_idx.mask]
    return indices_from, indices_to


def match_path(
    kpts_graph: nx.Graph,
    kpts_path: tuple[str, str, str],
    return_kpts: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    start_from, intermediary_from = kpts_path[0], kpts_path[1]
    intermediary_to, end_to = kpts_path[1], kpts_path[2]
    edge_from = kpts_graph[start_from][intermediary_from]
    edge_to = kpts_graph[intermediary_to][end_to]

    two_view_from: TwoViewEdge = edge_from["two_view"]
    two_view_to: TwoViewEdge = edge_to["two_view"]

    kpts_from, kpts_intermediary_from = (
        two_view_from.kpts_for[start_from],
        two_view_from.kpts_for[intermediary_from],
    )
    kpts_intermediary_to, kpts_to = (

        two_view_to.kpts_for[intermediary_to],
        two_view_to.kpts_for[end_to],
    )

    indices_from, indices_to = match_kpts(
        kpts_intermediary_from,
        kpts_intermediary_to,
    )
    if return_kpts:
        return kpts_from, kpts_to, indices_from, indices_to

    return indices_from, indices_to


def merge_path(
    kpts_graph: nx.Graph,
    kpts_path: tuple[str, str, str],
    min_matches: int = 500,
) -> TwoViewEdge:
    kpts_from, kpts_to, indices_from, indices_to = match_path(
        kpts_graph,
        kpts_path,
        return_kpts=True,
    )

    if len(indices_from) < min_matches:
        return None
    node_from, via, node_to = kpts_path
    from_matched_kpts = kpts_from[indices_from]
    to_matched_kpts = kpts_to[indices_to]

    inlier_indices = validate_kps_matches(
        kpts_from[indices_from],
        kpts_to[indices_to],
        kpts_graph.nodes[node_from]["image"].hw,
        kpts_graph.nodes[node_to]["image"].hw,
    )
    if len(inlier_indices) < min_matches:
        return None
    from_inlier_kpts = from_matched_kpts[inlier_indices[:, 0]]
    to_inlier_kpts = to_matched_kpts[inlier_indices[:, 1]]

    return TwoViewEdge(
            st_filepath=node_from,
            nd_filepath=node_to,
            kpts_for={
                node_from: from_inlier_kpts,
                node_to: to_inlier_kpts,
            },
            match_kind=MatchKind.MERGED,
            num_matches=len(from_inlier_kpts)
        )


In [ ]:
scene_graph = nx.Graph().to_undirected()
for image_id, image in image_repository.iterate_over_images():
    height, width = image.shape[:2]
    image = Image(
        height=height,
        width=width,
    )
    scene_graph.add_node(
        image_repository.get_filepath(image_id),
        image=image,
    )

In [ ]:
for st_filepath, matched_filepaths_map in matches_map.items():
    for nd_filepath, kpts in matched_filepaths_map.items():
        st_kpts, nd_kpts = np.split(kpts, 2,axis=1)
        scene_graph.add_edge(
            st_filepath,
            nd_filepath,
            two_view=TwoViewEdge(
                st_filepath=st_filepath,
                nd_filepath=nd_filepath,
                kpts_for={
                    st_filepath: st_kpts,
                    nd_filepath: nd_kpts,
                },
                match_kind=MatchKind.MATCHED,
                num_matches=len(kpts)
            ),
            weight=len(kpts),
        )

In [ ]:
plot_graph(scene_graph)

In [ ]:
import itertools as it

In [ ]:
pairs = list(it.combinations(filepaths_as_str, 2))

In [ ]:
import pycolmap

In [ ]:
from tqdm.auto import tqdm
from mts.pipeline.repository.base import BaseImageRepository

In [ ]:
def match_them(
    st_fpath: str,
    nd_fpath: str,
    scene_graph: nx.Graph,
    model,
    device,
) -> None:
    matches_map = extract_dense_keypoints(
        model,
        [(0, 1)],
        [st_fpath, nd_fpath],
        device=device,
    )
    try_first = st_fpath
    try_second = nd_fpath

    if len(matches_map) == 0:
        return

    try:
        matches_mmap = matches_map[try_first]
    except KeyError:
        matches_mmap = matches_map[try_second]
        try_first, try_second = try_second, try_first

    if len(matches_mmap) == 0:
        return

    matches = matches_mmap[try_second]
    st_kpts, nd_kpts = np.split(matches, 2, axis=1)
    if len(matches) >= 500:

        scene_graph.add_edge(
            try_first,
            try_second,
            two_view=TwoViewEdge(
                st_filepath=try_first,
                nd_filepath=try_second,
                kpts_for={
                    try_first: st_kpts,
                    try_second: nd_kpts,
                },
                match_kind=MatchKind.MATCHED,
                num_matches=len(matches),
            ),
            weight=len(matches),
        )

In [ ]:
from collections import Counter


def nums(scene_graph):
    return Counter(
        [
            edge_data["two_view"].match_kind.value
            for edge_data in scene_graph.edges.values()
        ]
    )

In [ ]:
def do_something(
    kpts_graph: nx.Graph,
    pairs: list[tuple[str, str]],
    device,
    model,
):
    num_merge = 0
    for st_fpath, nd_fpath in tqdm(pairs):
        if not kpts_graph.has_node(st_fpath):
            print(f"`{st_fpath}` not in the graph")
            continue
        if not kpts_graph.has_node(nd_fpath):
            print(f"`{nd_fpath}` not in the graph")
            continue
        merged = False
        st_fpath, nd_fpath = sorted([st_fpath, nd_fpath])

        if not kpts_graph.has_edge(st_fpath, nd_fpath):
            for kpts_path in list(
                nx.all_simple_paths(kpts_graph, st_fpath, nd_fpath, cutoff=2)
            ):
                # TODO: Add a check if there are enough merges, if so, then merge, otherwise compute the matchings - which is more costly
                # TODO: Check if it can be reconstructed from different paths
                two_view: TwoViewEdge | None = merge_path(kpts_graph, kpts_path)
                if two_view is not None and two_view.num_matches > 500:
                    before_count = nums(kpts_graph)

                    node_from, via, node_to = kpts_path
                    kpts_graph.add_edge(
                        node_from,
                        node_to,
                        two_view=two_view,
                        weight=two_view.num_matches,
                    )
                    after_count = nums(kpts_graph)

                    print(f"merge new edge ({node_from}, {node_to}) via `{via}`; before count {before_count}; after count {after_count}")
                    merged = True
                    num_merge += 1
                    break
        else:
            merged = True
        if not merged:
            match_them(st_fpath, nd_fpath, kpts_graph, model, device)
    return num_merge

In [ ]:
possible_pairs_as_str = []
for st_idx, nd_idx in possible_pairs:
    st_idx, nd_idx = sorted((st_idx, nd_idx))

    possible_pairs_as_str.append(
        (
            idx_to_filepath_as_str_map[st_idx],
            idx_to_filepath_as_str_map[nd_idx],
        )
    )

In [ ]:
num_merged = do_something(
    scene_graph,
    possible_pairs_as_str,
    device=cuda0,
    model=mast3r_model,
)

In [ ]:
num_merged

In [ ]:
len(scene_graph.edges)

In [ ]:
plot_graph(scene_graph)

In [ ]:
from mts.core.matcher.dense.merge.round import merge_matches

In [ ]:
import pandas as pd

In [ ]:
from collections import Counter

In [ ]:
Counter([edge_data['two_view'].match_kind.value for edge_data in scene_graph.edges.values()])

In [ ]:
matches_dict = {}
for (img1, img2), edge_data in scene_graph.edges.items():
    two_view: TwoViewEdge = edge_data["two_view"]
    matches = np.concatenate(
        [
            two_view.kpts_for[img1],
            two_view.kpts_for[img2],
        ],
        axis=1,
    )
    matches_dict.setdefault(img1, {})[img2] = matches

In [ ]:
global_keypoints, global_matches = merge_matches(matches_dict)

In [ ]:
match_no = 1
st_fpath, nd_fpath = list(global_matches.keys())[match_no]

In [ ]:
matches = global_matches[st_fpath, nd_fpath]

In [ ]:
st_image_id = image_repository.get_image_id(st_fpath)
nd_image_id = image_repository.get_image_id(nd_fpath)

st_image = image_repository.load_image(st_image_id)
nd_image = image_repository.load_image(nd_image_id)

st_kpts = global_keypoints[st_fpath]
nd_kpts = global_keypoints[nd_fpath]

In [ ]:
img_viz.plot_images([st_image, nd_image])

In [ ]:
img_viz.plot_images([st_image, nd_image])
img_viz.plot_keypoints([st_kpts, nd_kpts])
img_viz.plot_matches(
    st_kpts[matches[:, 0]],
    nd_kpts[matches[:, 1]],
    a=0.1,
)